In [ ]:
# ── Cell 0: Mount Drive & Unzip Dataset ───────────────────────────────────────
# Upload heritagelens-data.zip (656 MB) to Google Drive → MyDrive, then run.
# Extracts to /content/images/<filename>.jpg + /content/metadata_merged.json
# ──────────────────────────────────────────────────────────────────────────────

from google.colab import drive
import os, shutil, sys, json
from pathlib import Path

drive.mount("/content/drive", force_remount=True)

zip_name   = "heritagelens-data.zip"
candidates = [Path("/content/drive/MyDrive") / zip_name, Path("/content") / zip_name]
target_zip = next((p for p in candidates if p.exists()), None)

if target_zip:
    print(f"Found: {target_zip}  ({target_zip.stat().st_size/1e6:.1f} MB)")
    print("Unzipping ...")
    shutil.unpack_archive(str(target_zip), "/content/", "zip")
    print("Done.")
else:
    raise FileNotFoundError("heritagelens-data.zip not found. Upload to Drive → MyDrive.")

ROOT = Path("/content")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# ── Verify ─────────────────────────────────────────────────────────────────────
# Zip extracts flat: /content/metadata_merged.json + /content/images/<file>.jpg
MERGED_PATH = ROOT / "metadata_merged.json"
IMAGES_DIR  = ROOT / "images"

with open(MERGED_PATH) as f:
    meta = json.load(f)

img_count = sum(1 for f in IMAGES_DIR.iterdir() if f.is_file()) if IMAGES_DIR.exists() else 0

ext_n = sum(1 for e in meta if e.get("image_type") == "exterior")
obj_n = sum(1 for e in meta if e.get("image_type") == "object")

print(f"\nROOT              : {ROOT}")
print(f"metadata entries  : {len(meta)}  (exterior={ext_n}, object={obj_n})")
print(f"images on disk    : {img_count}")
print(f"unique monuments  : {len(set(e['monument_name'] for e in meta))}")

if img_count < len(meta) * 0.95:
    print("WARNING: fewer images on disk than metadata entries — check the zip.")
else:
    print("\n✓ All good — run the next cells in order.")

In [ ]:
# ── Cell 1: Dataset Preview ────────────────────────────────────────────────────
import json, random, unicodedata
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

ROOT        = Path("/content")
MERGED_PATH = ROOT / "metadata_merged.json"   # flat in zip root
IMAGES_DIR  = ROOT / "images"                 # flat images/ folder

with open(MERGED_PATH) as f:
    metadata = json.load(f)

cap1_lens = [len(e["captions"][0].split()) for e in metadata]
cap2_lens = [len(e["captions"][1].split()) for e in metadata]
ext_n = sum(1 for e in metadata if e.get("image_type") == "exterior")
obj_n = sum(1 for e in metadata if e.get("image_type") == "object")
print(f"Total images       : {len(metadata)}  (exterior={ext_n}, object={obj_n})")
print(f"Unique monuments   : {len(set(e['monument_name'] for e in metadata))}")
print(f"Avg Cap1 length    : {sum(cap1_lens)/len(cap1_lens):.1f} words")
print(f"Avg Cap2 length    : {sum(cap2_lens)/len(cap2_lens):.1f} words")

# Build NFC-normalized flat lookup: filename → path
def _nfc(s): return unicodedata.normalize("NFC", s)
_lookup = {}
if IMAGES_DIR.exists():
    for f in IMAGES_DIR.iterdir():
        if f.is_file():
            _lookup[_nfc(f.name)] = f

matched = sum(1 for e in metadata if _nfc(e["image_id"]) in _lookup)
missing = len(metadata) - matched
print(f"\nImages on disk     : {len(_lookup)}")
print(f"Matched to metadata: {matched}/{len(metadata)}")
if missing:
    print(f"MISSING {missing} — re-upload the latest zip (656 MB)")
else:
    print("✓ All images found")

# Preview grid
samples = random.sample(metadata, min(6, len(metadata)))
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, e in zip(axes.flatten(), samples):
    path = _lookup.get(_nfc(e["image_id"]))
    if path:
        ax.imshow(mpimg.imread(str(path)))
    else:
        ax.text(0.5, 0.5, "not found", ha="center", va="center", fontsize=9, color="red")
    ax.axis("off")
    c1 = e["captions"][0][:75] + "..." if len(e["captions"][0]) > 75 else e["captions"][0]
    c2 = e["captions"][1][:75] + "..." if len(e["captions"][1]) > 75 else e["captions"][1]
    ax.set_title(f"[{e.get('image_type','?')}]  {e['monument_name'][:38]}\n"
                 f"Cap1: {c1}\nCap2: {c2}", fontsize=6.5, loc="left")

n_monuments = len(set(e['monument_name'] for e in metadata))
plt.suptitle(f"HeritageLens — {len(metadata)} images, {n_monuments} monuments", fontsize=12)
plt.tight_layout(); plt.show()


In [ ]:
# ── Cell 2: Install Libraries & Global Setup ──────────────────────────────────
# Using BLIP (Bootstrapping Language-Image Pre-training) — pre-trained on
# 129M image-caption pairs, fine-tuned here on 530 Nepali heritage images.
# Far more effective than ResNet+GPT2 from scratch on small datasets.
# ──────────────────────────────────────────────────────────────────────────────
!pip install -q transformers>=4.36 torch torchvision tqdm nltk pillow

import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import torch
from transformers import BlipProcessor, BlipForConditionalGeneration
from tqdm import tqdm

ROOT = Path("/content")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"ROOT   = {ROOT}")
print(f"Device = {device}")
if device.type == "cuda":
    print(f"GPU    = {torch.cuda.get_device_name(0)}")
    print(f"VRAM   = {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Load BLIP processor (handles image preprocessing + tokenization) ──────────
MODEL_ID  = "Salesforce/blip-image-captioning-base"
processor = BlipProcessor.from_pretrained(MODEL_ID)
print(f"\nBLIP processor loaded  : {MODEL_ID}")
print("Ready — run Cell 3 next.")

In [ ]:
# ── Cell 3: Dataset & DataLoaders (BLIP) ──────────────────────────────────────
import json, random, unicodedata
from pathlib import Path
from PIL import Image
from PIL import ImageEnhance, ImageOps
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

def _nfc(s): return unicodedata.normalize("NFC", s)

# Build NFC-normalized filename → path lookup once
# Zip extracts flat: /content/images/<filename>.jpg  (no subdirectories)
_IMAGE_CACHE = {}
def _build_cache(images_dir):
    for f in Path(images_dir).iterdir():
        if f.is_file():
            _IMAGE_CACHE.setdefault(_nfc(f.name), f)

# ── Augmentation for training images (applied to PIL before BLIP processor) ──
_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([transforms.ColorJitter(0.3, 0.3, 0.2, 0.05)], p=0.6),
    transforms.RandomApply([transforms.GaussianBlur(3)], p=0.2),
    transforms.RandomResizedCrop(384, scale=(0.8, 1.0), ratio=(0.9, 1.1)),
])

class HeritageDataset(Dataset):
    """Returns (PIL_image, caption_str) for BLIP fine-tuning.
    Randomly samples Cap1 or Cap2 each step — no Cap3 (templated)."""

    def __init__(self, json_path, images_dir, augment=False):
        if not _IMAGE_CACHE:
            _build_cache(images_dir)
        with open(json_path) as f:
            meta = json.load(f)
        self.augment = augment
        self.entries = []
        skipped = 0
        for e in meta:
            path = _IMAGE_CACHE.get(_nfc(e["image_id"]))
            if path:
                self.entries.append((e, path))
            else:
                skipped += 1
        if skipped:
            print(f"  Skipped {skipped} entries (image not found)")

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        entry, path = self.entries[idx]
        # Sample Cap1 or Cap2 only (Cap3 is templated — excluded)
        caption = random.choice(entry["captions"][:2])
        img = Image.open(path).convert("RGB")
        if self.augment:
            img = _aug(img)
        return img, caption

# ── Hyperparameters ───────────────────────────────────────────────────────────
BATCH_SIZE   = 16    # fits T4 16GB comfortably; lower to 8 if OOM
MAX_CAP_LEN  = 50
VAL_FRACTION = 0.12  # ~64 val images
LR           = 2e-5  # low LR for fine-tuning a pretrained model
NUM_EPOCHS   = 30
PATIENCE     = 6
SEED         = 42

CKPT_DIR  = ROOT / "outputs/checkpoints"
FIG_DIR   = ROOT / "outputs/figures"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── Load full dataset ─────────────────────────────────────────────────────────
print("Loading dataset ...")
full_ds = HeritageDataset(
    json_path   = ROOT / "metadata_merged.json",  # flat in zip root
    images_dir  = ROOT / "images",                # flat images/ folder
    augment     = False,
)
print(f"Valid images: {len(full_ds)}")

n_val   = max(int(len(full_ds) * VAL_FRACTION), 1)
n_train = len(full_ds) - n_val
train_sub, val_sub = random_split(
    full_ds, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED),
)

# Wrap train split with augmentation
class AugSubset(Dataset):
    def __init__(self, subset):
        self.subset = subset
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        img, cap = self.subset[idx]
        return _aug(img), cap

train_ds = AugSubset(train_sub)
val_ds   = val_sub
print(f"Train: {n_train}  |  Val: {n_val}")

# ── Collate: processor handles image preprocessing + tokenization ─────────────
def collate_fn(batch):
    images   = [item[0] for item in batch]
    captions = [item[1] for item in batch]
    enc = processor(
        images=images,
        text=captions,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=MAX_CAP_LEN,
    )
    labels = enc["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    return enc["pixel_values"], enc["input_ids"], enc["attention_mask"], labels

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}")
print(f"Batch size: {BATCH_SIZE}  |  LR: {LR}")
print("✓ DataLoaders ready — run Cell 4 next.")

In [ ]:
# ── Cell 4: Load BLIP + Optimizer + Scheduler ─────────────────────────────────
# BLIP-base: ViT-B/16 vision encoder + BERT-based text decoder
# Pre-trained on 129M pairs → fine-tuned on 530 Nepal heritage images
# ──────────────────────────────────────────────────────────────────────────────
from transformers import BlipForConditionalGeneration
import torch

MODEL_ID = "Salesforce/blip-image-captioning-base"
model    = BlipForConditionalGeneration.from_pretrained(MODEL_ID).to(device)

# ── Freeze the vision encoder — only fine-tune the text decoder ───────────────
# With only 530 images, fine-tuning the entire ViT is likely to overfit.
# The vision encoder already generalises well; we only need domain-specific text.
for param in model.vision_model.parameters():
    param.requires_grad = False

total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_p    = total_p - trainable_p

print(f"Model        : {MODEL_ID}")
print(f"Total params : {total_p/1e6:.1f}M")
print(f"Trainable    : {trainable_p/1e6:.1f}M  (text decoder + cross-attention)")
print(f"Frozen       : {frozen_p/1e6:.1f}M  (ViT vision encoder)")

# ── Optimizer ─────────────────────────────────────────────────────────────────
# Low LR for fine-tuning; slightly higher for the projection layer
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, weight_decay=0.01,
)

# Warmup 2 epochs then cosine decay
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR
warmup    = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=2)
cosine    = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS - 2, eta_min=1e-7)
scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[2])

scaler = torch.amp.GradScaler("cuda")

print(f"\nOptimizer : AdamW  LR={LR}  weight_decay=0.01")
print(f"Scheduler : LinearWarmup(2) → CosineAnnealing")
print(f"Scaler    : AMP mixed precision")
print(f"\n✓ Model ready — run Cell 5 (Training) next.")

In [ ]:
# ── Cell 5: Fine-tuning Loop ───────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np
import nltk, random as _rnd
nltk.download("punkt_tab", quiet=True)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# ── BLEU generation helper ────────────────────────────────────────────────────
@torch.no_grad()
def _generate(pil_image, max_length=50, num_beams=4):
    """Generate caption from a PIL image using BLIP beam search."""
    model.eval()
    pv = processor(images=pil_image, return_tensors="pt").pixel_values.to(device)
    ids = model.generate(
        pixel_values=pv,
        max_length=max_length,
        num_beams=num_beams,
        early_stopping=True,
    )
    return processor.decode(ids[0], skip_special_tokens=True)

# ── Training ──────────────────────────────────────────────────────────────────
MAX_GRAD_NORM = 1.0
BLEU_SAMPLE_N = 20
history       = {"train_loss": [], "val_loss": [], "bleu": [], "lr": []}
best_val_loss = float("inf")
patience_ctr  = 0
smoother      = SmoothingFunction().method1

gpu_name = torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU"
print(f"Fine-tuning BLIP on {gpu_name}")
print(f"  Epochs={NUM_EPOCHS}  batch={BATCH_SIZE}  LR={LR}  patience={PATIENCE}")
print(f"  Train batches/epoch: {len(train_loader)}  |  Val batches: {len(val_loader)}")
print("-" * 60)

for epoch in range(NUM_EPOCHS):
    lr_now = optimizer.param_groups[0]["lr"]
    history["lr"].append(lr_now)

    # ── Train ──────────────────────────────────────────────────────────────────
    model.train()
    total_train = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:2d}/{NUM_EPOCHS} [train]")
    for pixel_values, input_ids, attention_mask, labels in pbar:
        pixel_values  = pixel_values.to(device)
        input_ids     = input_ids.to(device)
        attention_mask= attention_mask.to(device)
        labels        = labels.to(device)

        optimizer.zero_grad()
        with torch.amp.autocast("cuda"):
            out  = model(pixel_values=pixel_values,
                         input_ids=input_ids,
                         attention_mask=attention_mask,
                         labels=labels)
            loss = out.loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        total_train += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{lr_now:.1e}")

    avg_train = total_train / len(train_loader)
    history["train_loss"].append(avg_train)

    # ── Validate (loss) ────────────────────────────────────────────────────────
    model.eval()
    total_val = 0.0
    with torch.no_grad():
        for pixel_values, input_ids, attention_mask, labels in tqdm(
                val_loader, desc=f"Epoch {epoch+1:2d}/{NUM_EPOCHS} [val]  "):
            pixel_values   = pixel_values.to(device)
            input_ids      = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels         = labels.to(device)
            with torch.amp.autocast("cuda"):
                out = model(pixel_values=pixel_values,
                            input_ids=input_ids,
                            attention_mask=attention_mask,
                            labels=labels)
            total_val += out.loss.item()
    avg_val = total_val / len(val_loader)
    history["val_loss"].append(avg_val)

    # ── BLEU on N val images (beam search generation) ─────────────────────────
    bleu_scores = []
    sample_idx  = _rnd.sample(range(len(val_sub)), min(BLEU_SAMPLE_N, len(val_sub)))
    for si in sample_idx:
        entry, path = full_ds.entries[val_sub.indices[si]]
        ref_caps  = entry["captions"][:2]
        pil_img   = Image.open(path).convert("RGB")
        gen       = _generate(pil_img)
        gen_toks  = gen.split()
        ref_toks  = [c.split() for c in ref_caps]
        if gen_toks:
            bleu_scores.append(
                sentence_bleu(ref_toks, gen_toks,
                              weights=(0.25,0.25,0.25,0.25),
                              smoothing_function=smoother))
    avg_bleu = float(np.mean(bleu_scores)) if bleu_scores else 0.0
    history["bleu"].append(avg_bleu)

    print(f"Epoch {epoch+1:2d}/{NUM_EPOCHS}  "
          f"lr={lr_now:.1e}  train={avg_train:.4f}  "
          f"val={avg_val:.4f}  BLEU-4={avg_bleu:.4f}")

    # ── Checkpoint + early stopping ────────────────────────────────────────────
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        patience_ctr  = 0
        model.save_pretrained(CKPT_DIR / "best_model")
        processor.save_pretrained(CKPT_DIR / "best_model")
        print(f"  → Best model saved  (val={best_val_loss:.4f})")
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"  Early stopping at epoch {epoch+1}")
            break

    scheduler.step()

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ep = range(1, len(history["train_loss"]) + 1)
axes[0].plot(ep, history["train_loss"], label="Train"); axes[0].plot(ep, history["val_loss"], label="Val")
axes[0].set_title("Loss"); axes[0].legend()
axes[1].plot(ep, history["bleu"], color="green"); axes[1].set_title("BLEU-4")
axes[2].plot(ep, history["lr"],  color="orange"); axes[2].set_title("LR")
plt.tight_layout(); plt.savefig(FIG_DIR / "training_curves.png", dpi=150); plt.show()
print(f"\nDone. Best val loss: {best_val_loss:.4f}")
print(f"Best model saved to: {CKPT_DIR / 'best_model'}")

In [ ]:
# ── Cell 6: Inference & Evaluation (Fine-tuned BLIP) ─────────────────────────
import nltk
nltk.download("punkt", quiet=True)
nltk.download("wordnet", quiet=True)

from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
import random as _rnd
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score

smoother = SmoothingFunction().method1

# ── Load best checkpoint ───────────────────────────────────────────────────────
best_dir = CKPT_DIR / "best_model"
if best_dir.exists():
    model     = BlipForConditionalGeneration.from_pretrained(best_dir).to(device)
    processor = BlipProcessor.from_pretrained(best_dir)
    print(f"Loaded best model from: {best_dir}")
else:
    print("No checkpoint found — using current model weights.")
model.eval()

# ── Caption generation helper ─────────────────────────────────────────────────
@torch.no_grad()
def generate_caption(pil_image, mdl=None, proc=None, num_beams=4, max_length=60):
    mdl  = mdl  or model
    proc = proc or processor
    pv   = proc(images=pil_image, return_tensors="pt").pixel_values.to(device)
    ids  = mdl.generate(pixel_values=pv, max_length=max_length,
                        num_beams=num_beams, length_penalty=1.0,
                        no_repeat_ngram_size=3, early_stopping=True)
    return proc.decode(ids[0], skip_special_tokens=True)

# ── Shared metric helper ──────────────────────────────────────────────────────
def compute_all_metrics(gt_caps_list, gen_list):
    """
    gt_caps_list : list of list-of-strings (references per image)
    gen_list     : list of strings (hypothesis per image)
    Returns dict with bleu1..4 and meteor.
    """
    smoother = SmoothingFunction().method1
    b1, b2, b3, b4, met = [], [], [], [], []
    list_of_refs = []
    list_of_hyps = []
    for gt_caps, gen in zip(gt_caps_list, gen_list):
        refs     = [c.split() for c in gt_caps]
        gen_toks = gen.split()
        if not gen_toks:
            continue
        list_of_refs.append(refs)
        list_of_hyps.append(gen_toks)
        b1.append(sentence_bleu(refs, gen_toks, weights=(1,0,0,0),          smoothing_function=smoother))
        b2.append(sentence_bleu(refs, gen_toks, weights=(0.5,0.5,0,0),      smoothing_function=smoother))
        b3.append(sentence_bleu(refs, gen_toks, weights=(1/3,1/3,1/3,0),    smoothing_function=smoother))
        b4.append(sentence_bleu(refs, gen_toks, weights=(0.25,0.25,0.25,0.25), smoothing_function=smoother))
        met.append(meteor_score([c.split() for c in gt_caps], gen_toks))
    return {
        "bleu1":  float(np.mean(b1))   if b1  else 0.0,
        "bleu2":  float(np.mean(b2))   if b2  else 0.0,
        "bleu3":  float(np.mean(b3))   if b3  else 0.0,
        "bleu4":  float(np.mean(b4))   if b4  else 0.0,
        "meteor": float(np.mean(met))  if met else 0.0,
        "n":      len(b1),
    }

# ── Inference grid: 8 val images ──────────────────────────────────────────────
N_SHOW     = 8
sample_idx = _rnd.sample(list(val_sub.indices), min(N_SHOW, len(val_sub)))

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
for ax, ds_idx in zip(axes.flatten(), sample_idx):
    entry, path = full_ds.entries[ds_idx]
    gt_caps     = entry["captions"][:2]
    img         = Image.open(path).convert("RGB")
    gen         = generate_caption(img)
    ax.imshow(img); ax.axis("off")
    gt_str  = gt_caps[1][:80] + "..." if len(gt_caps[1]) > 80 else gt_caps[1]
    gen_str = gen[:90] + "..." if len(gen) > 90 else gen
    itype   = entry.get("image_type", "")
    ax.set_title(f"[{itype}] {entry['monument_name'][:35]}\n"
                 f"GT : {gt_str}\nGEN: {gen_str}", fontsize=6.5, loc="left")

plt.suptitle("Fine-tuned BLIP — Validation Sample", fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / "inference_grid.png", dpi=150); plt.show()

# ── Full validation evaluation ────────────────────────────────────────────────
print(f"\nEvaluating fine-tuned BLIP on full val set ({len(val_sub)} images) ...")
ft_refs, ft_gens = [], []
for vi in tqdm(val_sub.indices, desc="Fine-tuned BLIP"):
    entry, path = full_ds.entries[vi]
    img         = Image.open(path).convert("RGB")
    ft_refs.append(entry["captions"][:2])
    ft_gens.append(generate_caption(img))

ft_metrics = compute_all_metrics(ft_refs, ft_gens)

print("\n" + "="*55)
print(f"  FINE-TUNED BLIP  —  {ft_metrics['n']} images")
print("="*55)
print(f"  BLEU-1  : {ft_metrics['bleu1']:.4f}")
print(f"  BLEU-2  : {ft_metrics['bleu2']:.4f}")
print(f"  BLEU-3  : {ft_metrics['bleu3']:.4f}")
print(f"  BLEU-4  : {ft_metrics['bleu4']:.4f}")
print(f"  METEOR  : {ft_metrics['meteor']:.4f}")
print("="*55)
print("Expected BLEU-4 range for fine-tuned BLIP: 0.10 – 0.25")

# Store for comparison in Cell 7
import json as _json
(FIG_DIR / "ft_metrics.json").write_text(_json.dumps(ft_metrics, indent=2))


In [ ]:
# ── Cell 7: Zero-shot BLIP Baseline ──────────────────────────────────────────
# Loads pristine blip-image-captioning-base (no fine-tuning) and evaluates
# on the same val set to measure how much fine-tuning actually helped.
import json as _json

print("Loading zero-shot BLIP (no fine-tuning) ...")
zs_model     = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base").to(device)
zs_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
zs_model.eval()
print("Zero-shot BLIP loaded.")

# ── Evaluate zero-shot on same val set ────────────────────────────────────────
print(f"\nEvaluating zero-shot BLIP on full val set ({len(val_sub)} images) ...")
zs_refs, zs_gens = [], []
for vi in tqdm(val_sub.indices, desc="Zero-shot BLIP"):
    entry, path = full_ds.entries[vi]
    img         = Image.open(path).convert("RGB")
    zs_refs.append(entry["captions"][:2])
    zs_gens.append(generate_caption(img, mdl=zs_model, proc=zs_processor))

zs_metrics = compute_all_metrics(zs_refs, zs_gens)
(FIG_DIR / "zs_metrics.json").write_text(_json.dumps(zs_metrics, indent=2))

# ── Side-by-side comparison ───────────────────────────────────────────────────
ft_metrics = _json.loads((FIG_DIR / "ft_metrics.json").read_text())

print("\n" + "="*60)
print(f"  COMPARISON  —  {ft_metrics['n']} val images")
print("="*60)
print(f"  {'Metric':<10}  {'Zero-shot BLIP':>16}  {'Fine-tuned BLIP':>16}  {'Δ':>8}")
print("-"*60)
for m in ["bleu1", "bleu2", "bleu3", "bleu4", "meteor"]:
    zs  = zs_metrics[m]
    ft  = ft_metrics[m]
    delta = ft - zs
    sign  = "+" if delta >= 0 else ""
    print(f"  {m.upper():<10}  {zs:>16.4f}  {ft:>16.4f}  {sign}{delta:>7.4f}")
print("="*60)

improvement = ft_metrics["bleu4"] / zs_metrics["bleu4"] if zs_metrics["bleu4"] > 0 else float("inf")
print(f"\n  Fine-tuning improved BLEU-4 by {improvement:.1f}× over zero-shot BLIP")

In [ ]:
# ── Cell 8: Qualitative Comparison ───────────────────────────────────────────
# Shows 6 val images with Ground Truth, Fine-tuned BLIP, and Zero-shot BLIP
# captions side-by-side for a qualitative analysis.

import textwrap

N_QUAL  = 6
qual_idx = _rnd.sample(list(val_sub.indices), min(N_QUAL, len(val_sub)))

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.patch.set_facecolor("#FAFAFA")

for ax, ds_idx in zip(axes.flatten(), qual_idx):
    entry, path = full_ds.entries[ds_idx]
    gt_cap  = entry["captions"][1]  # Cap 2 — more readable
    img     = Image.open(path).convert("RGB")

    ft_cap  = generate_caption(img, mdl=model,    proc=processor)
    zs_cap  = generate_caption(img, mdl=zs_model, proc=zs_processor)

    ax.imshow(img)
    ax.axis("off")

    def wrap(text, width=42):
        return "\n".join(textwrap.wrap(text, width))

    itype = entry.get("image_type", "")
    title = (f"[{itype}] {entry['monument_name'][:38]}\n"
             f"GT : {wrap(gt_cap)}\n"
             f"FT : {wrap(ft_cap)}\n"
             f"ZS : {wrap(zs_cap)}")
    ax.set_title(title, fontsize=6.5, loc="left",
                 fontfamily="monospace",
                 bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.85))

plt.suptitle("Qualitative Comparison: Ground Truth vs Fine-tuned BLIP vs Zero-shot BLIP",
             fontsize=12, fontweight="bold")
plt.tight_layout()
out_path = FIG_DIR / "qualitative_comparison.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")
print("\nLegend:")
print("  GT = Ground truth caption (Cap 2)")
print("  FT = Fine-tuned BLIP (our model)")
print("  ZS = Zero-shot BLIP (no fine-tuning baseline)")